In [3]:
"""
Difference-in-Differences Analysis: Fire Effects on Recreation
=============================================================

Runs two-way fixed effects DiD to estimate how wildfires vs prescribed fires affect visitation. load_state_data prepares inputs and classify_treatment labels groups; estimate_fire_effect fits TWFE and returns monthly effects. aggregate_to_years collapses to annual windows and to_percent scales to percent; results are written to OUTPUT.

"""

import pandas as pd
import numpy as np
import re
from pathlib import Path  # added
 
# ============================================================
# YOUR DATA FILES
# ============================================================

# Resolve directory of this file (works in .py; falls back to CWD in notebooks)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

# Colorado data (contains predictions + features)
CO_DATA = (HERE / "../data/03_visitation/TreatCon_Pred_CO.csv").resolve()

# California data (contains predictions + features)
CA_DATA = (HERE / "../data/03_visitation/TreatCon_Pred_CA.csv").resolve()

# Output path (aligns with 04_DiD_FireType.ipynb expectation)
OUTPUT = (HERE / "../outputs/did_results_firetype.csv").resolve()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# ============================================================
# ANALYSIS SETTINGS
# ============================================================

MONTHS_PRE_FIRE = 12           # Include 12 months before fire
REFERENCE_MONTH = -2           # Set month -2 as baseline (=0)
MIN_SAMPLE_PERCENT = 0.10      # Drop months with <10% of max sample (months towards the 5-year mark with very little data)

# Control variables (only time-varying ones - site FE handles time-invariant)
CONTROLS = ['temperature_mean']  # Temperature is time-varying confounder that

# Note: Trail density, slope, vegetation are time-invariant and already 
# controlled for by site fixed effects. Including them would be redundant.

# ============================================================
# CLASSIFY TREATMENT STATUS
# ============================================================

def classify_treatment(treatment_type, state):
    """
    Classify sites as wildfire/prescribed fire × treated/control
    
    Returns: 'wildfire_treated', 'wildfire_control', 'rx_treated', 'rx_control'
    """
    if pd.isna(treatment_type):
        return 'unknown'
    
    tt = str(treatment_type).lower()
    
    if state == 'CO':
        # Colorado labels: "fire", "controlfire", "rx", "controlrx"
        if 'fire' in tt and 'control' not in tt:
            return 'wildfire_treated'
        elif 'controlfire' in tt:
            return 'wildfire_control'
        elif 'rx' in tt and 'control' not in tt:
            return 'rx_treated'
        elif 'controlrx' in tt:
            return 'rx_control'
    else:
        # California labels: "wildfire_treated", "wildfire_control", etc.
        if 'wildfire' in tt and 'treated' in tt:
            return 'wildfire_treated'
        elif 'wildfire' in tt and 'control' in tt:
            return 'wildfire_control'
        elif 'rx' in tt and 'treated' in tt:
            return 'rx_treated'
        elif 'rx' in tt and 'control' in tt:
            return 'rx_control'
    
    return 'unknown'

# ============================================================
# DiD ESTIMATION
# ============================================================

def estimate_fire_effect(data, fire_type, controls):
    """
    Estimate fire effects using Sun-Abraham saturated event study (PyFixest)
    
    Parameters:
    -----------
    data : DataFrame with columns [siteid, year, month, rel_month, outcome, treated, time_period]
    fire_type : 'wildfire' or 'prescribed'
    controls : list of control variable names
    
    Returns:
    --------
    monthly_effects : dict of {month: {'coef', 'se', 'ci_lower', 'ci_upper', 'n_treated'}}
    n_treated : number of treated sites
    n_control : number of control sites
    baseline : pre-fire average outcome for treated sites
    """
    try:
        import pyfixest as pf
    except ImportError:
        raise ImportError("Install pyfixest: pip install pyfixest")
    
    # Filter to relevant fire type
    if fire_type == 'wildfire':
        df = data[data['treatment_group'].isin(['wildfire_treated', 'wildfire_control'])].copy()
    else:
        df = data[data['treatment_group'].isin(['rx_treated', 'rx_control'])].copy()
    
    # Keep analysis window (12 months before fire onwards)
    df = df[df['rel_month'].notna()].copy()
    df = df[df['rel_month'] >= -MONTHS_PRE_FIRE].copy()
    df['rel_month'] = df['rel_month'].astype(int)
    
    # Count sites
    n_treated = df[df['treated'] == 1]['siteid'].nunique()
    n_control = df[df['treated'] == 0]['siteid'].nunique()
    
    if n_treated < 2 or n_control < 1:
        return None, None, None, None
    
    # Calculate baseline (pre-fire average for treated sites)
    baseline = df[(df['treated'] == 1) & (df['rel_month'] < 0)]['outcome'].mean()
    if pd.isna(baseline) or baseline == 0:
        baseline = df['outcome'].mean()
    
    # ============================================================
    # NEW: Sun-Abraham Preparation
    # ============================================================
    
    # Create absolute time variable (year_month)
    df['year_month'] = df['year'] * 12 + df['month']
    
    # Create cohort variable (first treatment time for treated units)
    treated_sites = df.loc[df['treated'] == 1, 'siteid'].unique()
    cohort_dict = {}
    
    for sid in treated_sites:
        site_data = df[df['siteid'] == sid]
        treatment_time = site_data.loc[site_data['rel_month'] == 0, 'year_month']
        if len(treatment_time) > 0:
            cohort_dict[sid] = int(treatment_time.iloc[0])
    
    df['cohort'] = df['siteid'].map(cohort_dict)
    
    # Never-treated units: cohort must be NaN (not 0!)
    df.loc[df['treated'] == 0, 'cohort'] = np.nan
    
    # Drop treated units without cohort assignment
    df = df[~((df['treated'] == 1) & (df['cohort'].isna()))].copy()
    
    if len(df) == 0:
        return None, None, None, None
    
    # Track n_treated per month for min-sample filtering
    n_treated_by_month = df[df['treated'] == 1].groupby('rel_month')['siteid'].nunique().to_dict()
    
    # ============================================================
    # NEW: Sun-Abraham Estimation
    # ============================================================
    
    # Prepare controls formula
    available_controls = [c for c in controls if c in df.columns]
    xfml = " + ".join(available_controls) if available_controls else None
    
    try:
        # Run saturated event study
        fit = pf.event_study(
            data=df,
            yname='outcome',
            idname='siteid',
            tname='year_month',
            gname='cohort',
            xfml=xfml,
            estimator='saturated'
        )
        
        # Aggregate with share-weighting
        agg = fit.aggregate(weighting='shares')
        
        # Extract monthly effects
        effects = {}
        for rel_time, row in agg.iterrows():
            month = int(round(float(rel_time)))
            coef = float(row['Estimate'])
            se = float(row['Std. Error'])
            
            effects[month] = {
                'coef': coef,
                'se': se,
                'ci_lower': coef - 1.96 * se,
                'ci_upper': coef + 1.96 * se,
                'n_treated': n_treated_by_month.get(month, 0)
            }
        
        # Add reference month (set to zero)
        if REFERENCE_MONTH not in effects:
            effects[REFERENCE_MONTH] = {
                'coef': 0.0, 'se': 0.0, 'ci_lower': 0.0, 'ci_upper': 0.0,
                'n_treated': n_treated_by_month.get(REFERENCE_MONTH, 0)
            }
        
        return dict(sorted(effects.items())), n_treated, n_control, baseline
        
    except Exception as e:
        print(f"    Sun-Abraham estimation failed: {str(e)}")
        return None, None, None, None


def aggregate_to_years(monthly_effects):
    """
    Aggregate monthly effects to annual using inverse-variance weighting
    
    Groups months into years:
    - Month 0-11 → Year 1
    - Month 12-23 → Year 2
    - Month 24-35 → Year 3
    - etc.
    
    Drops months where treated sample falls below 10% of peak.
    """
    if not monthly_effects:
        return None
    
    df = pd.DataFrame.from_dict(monthly_effects, orient='index')
    
    # Find minimum acceptable sample (10% of peak across all months)
    peak_sample = df['n_treated'].max()
    min_sample = max(1, int(np.floor(peak_sample * MIN_SAMPLE_PERCENT)))
    
    # Keep only post-fire months with sufficient sample
    df = df[df.index >= 0]
    df = df[df['n_treated'] >= min_sample]
    
    if len(df) == 0:
        return None
    
    # Assign months to years
    df['year'] = np.floor(df.index / 12).astype(int) + 1
    df.loc[df['year'] > 5, 'year'] = 5  # Cap at year 5+
    
    # Aggregate by year using inverse-variance weights
    annual = []
    for year, group in df.groupby('year'):
        # Weight by precision (1/variance)
        weights = 1 / (group['se']**2 + 1e-10)
        weights = weights / weights.sum()
        
        # Weighted average
        coef = (group['coef'] * weights).sum()
        se = 1 / np.sqrt((1 / (group['se']**2 + 1e-10)).sum())
        
        annual.append({
            'year': int(year),
            'coef': coef,
            'se': se,
            'ci_lower': coef - 1.96 * se,
            'ci_upper': coef + 1.96 * se
        })
    
    # Add year 0 (reference)
    annual.insert(0, {
        'year': 0, 'coef': 0.0, 'se': 0.0, 'ci_lower': 0.0, 'ci_upper': 0.0
    })
    
    return {row['year']: row for row in annual}


def to_percent(effects, baseline):
    """Convert raw coefficients to percentage changes"""
    if baseline == 0:
        return effects
    
    return {
        year: {
            'coef': (vals['coef'] / baseline) * 100,
            'se': (vals['se'] / baseline) * 100,
            'ci_lower': (vals['ci_lower'] / baseline) * 100,
            'ci_upper': (vals['ci_upper'] / baseline) * 100
        }
        for year, vals in effects.items()
    }

# ============================================================
# LOAD DATA
# ============================================================

print("=" * 60)
print("LOADING DATA")
print("=" * 60)

def load_state_data(data_file, controls, state_name):
    """Load data file (already has predictions + features merged)"""
    # Load file
    df = pd.read_csv(data_file)
    df.columns = df.columns.str.strip().str.lower()
    
    # Classify treatment status
    df['treatment_group'] = df['treatment_type'].apply(lambda x: classify_treatment(x, state_name))
    df['treated'] = df['treatment_group'].str.contains('treated').astype(int)
    
    # Create outcome and time period
    df['outcome'] = df['y_pred']
    df['time_period'] = df['year'].astype(str) + '_' + df['month'].astype(str).str.zfill(2)
    
    # Fill missing controls with median
    for col in controls:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    
    return df

# Load Colorado
print("\nColorado:")
co_data = load_state_data(CO_DATA, CONTROLS, 'CO')
print(f"  {len(co_data):,} observations, {co_data['siteid'].nunique()} sites")

# Load California
print("\nCalifornia:")
ca_data = load_state_data(CA_DATA, CONTROLS, 'CA')
print(f"  {len(ca_data):,} observations, {ca_data['siteid'].nunique()} sites")

# ============================================================
# RUN ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("ESTIMATING FIRE EFFECTS")
print("=" * 60)

results = []

for state, data in [('CO', co_data), ('CA', ca_data)]:
    print(f"\n{state}:")
    
    for fire_type in ['wildfire', 'prescribed']:
        print(f"  {fire_type}...", end=' ')
        
        # Estimate monthly effects
        monthly, n_treat, n_control, baseline = estimate_fire_effect(data, fire_type, CONTROLS)
        
        if monthly:
            # Aggregate to years
            annual = aggregate_to_years(monthly)
            
            if annual:
                # Convert to percentage changes
                annual_pct = to_percent(annual, baseline)
                
                # Save results
                for year, vals in annual_pct.items():
                    results.append({
                        'state': state,
                        'fire_type': fire_type,
                        'year': year,
                        'effect_pct': vals['coef'],
                        'ci_lower': vals['ci_lower'],
                        'ci_upper': vals['ci_upper'],
                        'n_treated': n_treat,
                        'n_control': n_control
                    })
                
                print(f"✓ ({n_treat} treated, {n_control} control)")
            else:
                print("insufficient data")
        else:
            print("failed")

# ============================================================
# SAVE & SUMMARIZE
# ============================================================

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)

# Save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT, index=False)
print(f"\n✓ Saved to: {OUTPUT}")

# Print Year 1 effects
print("\nYear 1 Post-Fire Effects:")
print("-" * 60)

year1 = df_results[df_results['year'] == 1]

for state in ['CO', 'CA']:
    state_name = 'Colorado' if state == 'CO' else 'California'
    print(f"\n{state_name}:")
    
    for fire_type in ['wildfire', 'prescribed']:
        row = year1[(year1['state'] == state) & (year1['fire_type'] == fire_type)]
        
        if len(row) > 0:
            row = row.iloc[0]
            print(f"  {fire_type.capitalize():12s}: {row['effect_pct']:+6.1f}% "
                  f"(95% CI: {row['ci_lower']:+6.1f}% to {row['ci_upper']:+6.1f}%)")
            print(f"                 n={row['n_treated']} treated, {row['n_control']} control")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

LOADING DATA

Colorado:
  8,160 observations, 136 sites

California:
  22,380 observations, 373 sites

ESTIMATING FIRE EFFECTS

CO:
  wildfire... 

C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\did\saturated_twfe.py:68: UserWarning: The SaturatedEventStudyClass is currently in beta. Please report any issues you may encounter.
  warnings.warn(
C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\estimation\feols_.py:2847: UserWarning: 
            548 variables dropped due to multicollinearity.
            The following variables are dropped: 
    C(rel_time, contr.treatment(base=-1.0))[-12.0]:cohort_dummy_24244
    C(rel_time, contr.treatment(base=-1.0))[-11.0]:cohort_dummy_24244
    C(rel_time, contr.treatment(base=-1.0))[-10.0]:cohort_dummy_24244
    C(rel_time, contr.treatment(base=-1.0))[-9.0]:cohort_dummy_24244
    C(rel_time, contr.treatment(base=-1.0))[-8.0]:cohort_dummy_24244
    ....
            
  warnings.warn(


✓ (44 treated, 40 control)
  prescribed... 

C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\did\saturated_twfe.py:68: UserWarning: The SaturatedEventStudyClass is currently in beta. Please report any issues you may encounter.
  warnings.warn(
C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\estimation\feols_.py:2847: UserWarning: 
            171 variables dropped due to multicollinearity.
            The following variables are dropped: 
    C(rel_time, contr.treatment(base=-1.0))[-12.0]:cohort_dummy_24243
    C(rel_time, contr.treatment(base=-1.0))[-11.0]:cohort_dummy_24243
    C(rel_time, contr.treatment(base=-1.0))[-10.0]:cohort_dummy_24243
    C(rel_time, contr.treatment(base=-1.0))[-9.0]:cohort_dummy_24243
    C(rel_time, contr.treatment(base=-1.0))[-8.0]:cohort_dummy_24243
    ....
            
  warnings.warn(


✓ (22 treated, 22 control)

CA:
  wildfire... 

C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\did\saturated_twfe.py:68: UserWarning: The SaturatedEventStudyClass is currently in beta. Please report any issues you may encounter.
  warnings.warn(
C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\estimation\feols_.py:2847: UserWarning: 
            678 variables dropped due to multicollinearity.
            The following variables are dropped: 
    C(rel_time, contr.treatment(base=-1.0))[-12.0]:cohort_dummy_24245
    C(rel_time, contr.treatment(base=-1.0))[-11.0]:cohort_dummy_24245
    C(rel_time, contr.treatment(base=-1.0))[-10.0]:cohort_dummy_24245
    C(rel_time, contr.treatment(base=-1.0))[-9.0]:cohort_dummy_24245
    C(rel_time, contr.treatment(base=-1.0))[-8.0]:cohort_dummy_24245
    ....
            
  warnings.warn(


✓ (141 treated, 142 control)
  prescribed... 

C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\did\saturated_twfe.py:68: UserWarning: The SaturatedEventStudyClass is currently in beta. Please report any issues you may encounter.
  warnings.warn(
C:\Users\kyma3609\AppData\Local\anaconda3_new\Lib\site-packages\pyfixest\estimation\feols_.py:2847: UserWarning: 
            278 variables dropped due to multicollinearity.
            The following variables are dropped: 
    C(rel_time, contr.treatment(base=-1.0))[-12.0]:cohort_dummy_24242
    C(rel_time, contr.treatment(base=-1.0))[-11.0]:cohort_dummy_24242
    C(rel_time, contr.treatment(base=-1.0))[-10.0]:cohort_dummy_24242
    C(rel_time, contr.treatment(base=-1.0))[-9.0]:cohort_dummy_24242
    C(rel_time, contr.treatment(base=-1.0))[-8.0]:cohort_dummy_24242
    ....
            
  warnings.warn(


✓ (32 treated, 35 control)

RESULTS

✓ Saved to: C:\Users\kyma3609\OneDrive - UCB-O365\Recreation_Fire\did_results.csv

Year 1 Post-Fire Effects:
------------------------------------------------------------

Colorado:
  Wildfire    :   -7.5% (95% CI:  -10.1% to   -4.9%)
                 n=44 treated, 40 control
  Prescribed  :   +8.3% (95% CI:   +5.0% to  +11.6%)
                 n=22 treated, 22 control

California:
  Wildfire    :  -18.0% (95% CI:  -22.5% to  -13.6%)
                 n=141 treated, 142 control
  Prescribed  :   -3.2% (95% CI:   -4.8% to   -1.5%)
                 n=32 treated, 35 control

DONE
